# 01 — Preprocessing (R / Seurat)

Loads the query Seurat object from `00_data_acquisition.ipynb`, applies QC filters, re-runs normalisation, HVG selection, PCA and saves the filtered object.


In [ ]:
# Papermill parameter cell — injected at runtime from config/params.yaml
min_genes    <- 200L
max_genes    <- 2500L
max_pct_mito <- 20
output_path  <- "data/cca/query.rds"
query_path   <- "data/cca/query.rds"


In [ ]:
suppressPackageStartupMessages({
  library(Seurat)
  library(ggplot2)
  library(patchwork)
  library(dplyr)
})
set.seed(42)
options(repr.plot.width = 9, repr.plot.height = 4)  # compact figures
cat(sprintf("Seurat %s | R %s\n", packageVersion("Seurat"), R.version$major))

## 1 · Load Data

In production: `Read10X(data.dir)` → `CreateSeuratObject()`.  
Here we load the built-in PBMC3k dataset from `SeuratData`.

In [ ]:
# Load query Seurat object produced by 00_data_acquisition.ipynb
stopifnot(file.exists(query_path))
seurat_obj <- readRDS(query_path)
cat(sprintf("Loaded: %d cells x %d genes  (from %s)\n",
            ncol(seurat_obj), nrow(seurat_obj), query_path))


## 2 · Quality Control

Flag mitochondrial genes (`^MT-`), calculate per-cell QC metrics, visualise and filter.

In [ ]:
seurat_obj[["percent.mt"]] <- PercentageFeatureSet(seurat_obj, pattern = "^MT-")

# Violin plots
VlnPlot(
  seurat_obj,
  features = c("nFeature_RNA", "nCount_RNA", "percent.mt"),
  ncol = 3, pt.size = 0.1
) + plot_annotation(title = "QC metrics — pre-filter")

In [ ]:
# Feature scatter: spot outliers
p1 <- FeatureScatter(seurat_obj, feature1 = "nCount_RNA", feature2 = "percent.mt") +
      geom_vline(xintercept = 0, linetype = "dashed") +
      ggtitle("UMI vs % mito")
p2 <- FeatureScatter(seurat_obj, feature1 = "nCount_RNA", feature2 = "nFeature_RNA") +
      geom_hline(yintercept = c(min_genes, max_genes), linetype = "dashed", colour = "red") +
      ggtitle("UMI vs genes per cell")
p1 + p2

In [ ]:
n_before <- ncol(seurat_obj)
seurat_obj <- subset(
  seurat_obj,
  subset = nFeature_RNA > min_genes &
           nFeature_RNA < max_genes &
           percent.mt  < max_pct_mito
)
cat(sprintf(
  "Cells before QC filter : %d\nCells after QC filter  : %d\nRemoved                : %d (%.1f%%)\n",
  n_before, ncol(seurat_obj), n_before - ncol(seurat_obj), (n_before - ncol(seurat_obj)) / n_before * 100
))

## 3 · Doublet Detection

In production use **DoubletFinder** (`install.packages('DoubletFinder')`) or **scDblFinder**.  
Here we simulate a doublet probability score (Beta(1,10)) and remove cells > 0.25.

In [ ]:
# --- Production replacement ---
# library(DoubletFinder)
# seurat_obj <- doubletFinder(seurat_obj, PCs = 1:30, pN = 0.25, pK = 0.09, nExp = ...)
# --------------------------------

set.seed(42)
doublet_score <- rbeta(ncol(seurat_obj), shape1 = 1, shape2 = 10)
seurat_obj$doublet_score     <- doublet_score
seurat_obj$predicted_doublet <- doublet_score > 0.25

n_dbl <- sum(seurat_obj$predicted_doublet)
cat(sprintf("Predicted doublets: %d (%.1f%%)\n", n_dbl, n_dbl / ncol(seurat_obj) * 100))

seurat_obj <- subset(seurat_obj, subset = predicted_doublet == FALSE)
cat(sprintf("Cells after doublet removal: %d\n", ncol(seurat_obj)))

## 4 · Normalisation & Log-transform

Library-size normalise to 10,000 counts per cell, then log1p-transform.

In [ ]:
seurat_obj <- NormalizeData(
  seurat_obj,
  normalization.method = "LogNormalize",
  scale.factor         = 1e4
)
cat("Normalisation complete (LogNormalize, scale.factor = 10,000)\n")

## 5 · Highly Variable Gene (HVG) Selection

In [ ]:
seurat_obj <- FindVariableFeatures(
  seurat_obj,
  selection.method = "vst",
  nfeatures        = 2000
)

top10 <- head(VariableFeatures(seurat_obj), 10)
cat(sprintf("HVGs selected: %d\nTop 10: %s\n",
            length(VariableFeatures(seurat_obj)), paste(top10, collapse = ", ")))

p_hvg <- VariableFeaturePlot(seurat_obj)
LabelPoints(plot = p_hvg, points = top10, repel = TRUE)

## 6 · Scaling & PCA

Scale HVGs (zero-mean, unit-variance), run PCA with 50 components.

In [ ]:
seurat_obj <- ScaleData(seurat_obj, features = VariableFeatures(seurat_obj))

seurat_obj <- RunPCA(
  seurat_obj,
  features = VariableFeatures(seurat_obj),
  npcs     = 50,
  verbose  = FALSE
)

cat("Top genes per PC:\n")
print(VizDimLoadings(seurat_obj, dims = 1:4, reduction = "pca"))

ElbowPlot(seurat_obj, ndims = 50) +
  ggtitle("PCA elbow plot — select dims for downstream analysis")

In [ ]:
# Examine top PCs
DimHeatmap(seurat_obj, dims = 1:6, cells = 500, balanced = TRUE)

## 7 · Save Pre-processed Seurat Object

In [ ]:
dir.create(dirname(output_path), showWarnings = FALSE, recursive = TRUE)
saveRDS(seurat_obj, file = output_path)

cat(sprintf("Saved Seurat object → %s\n", output_path))
cat(sprintf("Final dataset : %d cells x %d HVGs\n", ncol(seurat_obj), length(VariableFeatures(seurat_obj))))
cat(sprintf("PCA dims      : %d\n", ncol(Embeddings(seurat_obj, 'pca'))))

# Summary of object
seurat_obj